# Stage 1: Select Position Openers

Select wallets whose opening BUYs are worth copying.
Uses volatility-based wallet metrics + threshold scoring (matching reference notebook).

Grid-search over selection thresholds to maximize **copyable PnL from opening buys** on the validation split.

**Output:** `stage1_result.json` with best selection params.

In [14]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    evaluate_wallet_group,
    evaluate_wallet_group_openers,
    select_copyable_group,
    run_grid_search,
    save_stage_result,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [2]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full)

Markets: 1877548
Filtered markets for {'Weather'}: 87967
Loading 16 trade shards...
Total trades loaded: 13,603,198
Unique wallets: 4,054
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-22 05:47:10+00:00
Train:     6,619,486 trades  (24,504 markets)
Val:       4,081,660 trades  (16,109 markets)
Test:      2,902,052 trades  (13,119 markets)
Total:    13,603,198 trades  (53,732 markets)


## Compute wallet metrics on training data

In [3]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "opening_roi", "opening_pnl", "opening_copyable_pnl", "copyable_pnl", "num_buckets"]].head(10)

Wallets with metrics: 3514


,wallet,buy_roi,opening_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,num_buckets
0,0x00546dfeb4097e232c86a775ccbe3c9b84c0cab1,0.0236,0.0284,109.5627,0.8428,13.2971,45
1,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0046,0.0043,39.2184,-0.4680,-2.5788,304
2,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0193,0.0024,55.8989,-162.0518,84.8491,9771
3,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,-1.0000,-1.0000,0.0000,0.0000,2
4,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.6776,0.6033,59.3074,55.1701,115.4494,34
5,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,-0.0877,-17.1649,-21.0658,-20.9264,299
6,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0128,0.0164,12.3443,5.3319,28.7030,63
7,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0100,0.0052,93.4067,-5.6239,38.8680,5701
8,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.1991,0.2514,1861.2796,444.6215,-64.3190,1473
9,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.0036,0.0032,93.1017,-89.7735,-87.4111,3279


## Baseline selection (reference defaults)

In [15]:
copyable_group = select_copyable_group(
    wallet_vol,
    min_buy_roi=0.05,
    min_num_buckets=20,
    min_num_markets=15,
    max_drawdown_to_pnl=0.20,
    max_top_market_pnl_pct=0.25,
    max_market_pnl_hhi=0.30,
    min_total_notional=5_000,
    min_opening_roi=0.0,
    min_opening_pnl=0,
    min_opening_copyable_roi=0.0,
)
print(f"Copyable group: {len(copyable_group)} wallets")
show_cols = ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
             "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
copyable_group[[c for c in show_cols if c in copyable_group.columns]].head(15)

Copyable group: 48 wallets


,wallet,opening_roi,opening_copyable_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,buy_roi,num_buckets
0,0x74957ea27ac4fbdee46d861fdae357859ff67fcf,0.3587,0.7800,1048.9341,302.6835,858.2164,0.5852,13983
1,0xa89518aca5a633a79ad1e9737209c9689f83faac,1.2261,0.6654,6470.6572,326.1863,1073.0446,1.1189,561
2,0xd8798d9aa2c7c05bfefaf6436e8cbd4cd5de7bc3,0.6479,0.4740,709.5858,148.4676,266.7017,0.4606,559
3,0x41f623e0baae41e385875a3759c474cf0622828e,0.3993,0.4167,1572.4413,644.7670,3193.8278,0.5236,755
4,0x438ae660f4236553c74c776c8fdcf394912061c4,0.3228,0.4040,1032.1160,460.1703,1354.1948,0.4914,304
5,0x336e80318ab21fc5eb87434a2755b47a6eb3bcc3,0.4424,0.3830,509.9365,184.8350,191.0109,0.2158,534
6,0x177500541ae20bb0d46ab0db3fd2559e2a7e85b0,0.2225,0.3825,420.6573,296.3813,688.2982,0.1514,1278
7,0x0af29cc14fb231218c8d005685ac3db427a1ce86,0.1560,0.3563,858.8236,733.7985,2306.9008,0.1517,977
8,0x35aff83368c69c47af04ee2d99330154f22f1ca6,0.5797,0.3257,493.6992,140.3441,206.5448,0.6693,489
9,0x07d601375c9bbb9037ad3c7a8f8fa0deff8164fb,0.1161,0.2787,699.5974,288.4268,144.1246,0.0783,681


## Baseline evaluation (reference format)

In [16]:
wallet_set = set(copyable_group["wallet"])

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


*** TRAIN copyable group ***
  Open  : wallet_pnl=  64733.45  roi=0.1408  |  copyable_pnl=  13178.41  roi=0.1106
  Total : wallet_pnl= 156102.36  roi=0.0949  |  copyable_pnl=  40931.48  roi=0.0666

*** VAL copyable group ***
  Open  : wallet_pnl=  46939.30  roi=0.0948  |  copyable_pnl=   4984.52  roi=0.0395
  Total : wallet_pnl=  90704.32  roi=0.0595  |  copyable_pnl=   3617.37  roi=0.0076

*** TEST copyable group ***
  Open  : wallet_pnl=  33721.69  roi=0.0873  |  copyable_pnl=   5236.25  roi=0.0636
  Total : wallet_pnl=  56331.34  roi=0.0553  |  copyable_pnl=   4707.20  roi=0.0146


## Grid search

Vary selection thresholds to maximize copyable PnL from opening buys on the validation split.

In [73]:
param_grid = dict(
    min_buy_roi=[0, 0.05, 0.07],
    min_num_buckets=[15],
    min_num_markets=[10],
    max_drawdown_to_pnl=[0.20],
    max_top_market_pnl_pct=[1],
    max_market_pnl_hhi=[0.1, 0.30],
    min_total_notional=[1_000],
    min_opening_roi=[0.05, 0.07],
    min_opening_pnl=[200],
    min_opening_copyable_roi=[0.05],
)

print(f"Grid: {np.prod([len(v) for v in param_grid.values()]):.0f} combos")

Grid: 12 combos


In [74]:
res_df = run_grid_search(param_grid, wallet_vol, df_val)
# print(f'open_copyable_pnl: {res_df["open_copyable_pnl"].max():.0f}, open_copyable_roi: {res_df["open_copyable_pnl"].max():.4f}')
res_df.head()

best_row = res_df.iloc[0]
best_params = {k: best_row[k] for k in param_grid.keys()}
best_group = select_copyable_group(wallet_vol, **best_params)

print(f"Best config (val open copyable_pnl={best_row['open_copyable_pnl']:.2f}):")
print(best_params)
print(f"  wallets: {best_row['wallets']:.0f}  open_wallets: {best_row['open_wallets']:.0f}")

Grid: 12 combos, 8 workers
  [12/12] 1.3s elapsed
Done: 12 configs in 1.3s
Best config (val open copyable_pnl=4405.89):
{'min_buy_roi': np.float64(0.0), 'min_num_buckets': np.float64(15.0), 'min_num_markets': np.float64(10.0), 'max_drawdown_to_pnl': np.float64(0.2), 'max_top_market_pnl_pct': np.float64(1.0), 'max_market_pnl_hhi': np.float64(0.1), 'min_total_notional': np.float64(1000.0), 'min_opening_roi': np.float64(0.05), 'min_opening_pnl': np.float64(200.0), 'min_opening_copyable_roi': np.float64(0.05)}
  wallets: 45  open_wallets: 36


In [75]:
print('top 10 results')
res_df.head(10)

top 10 results


,min_buy_roi,min_num_buckets,min_num_markets,max_drawdown_to_pnl,max_top_market_pnl_pct,max_market_pnl_hhi,min_total_notional,min_opening_roi,min_opening_pnl,min_opening_copyable_roi,open_copyable_pnl,open_wallet_pnl,open_wallets,total_copyable_pnl,wallets,elapsed
1,0.0000,15,10,0.2000,1,0.1000,1000,0.0500,200,0.0500,4405.8925,43883.3564,36,7102.7763,45,0.8176
4,0.0500,15,10,0.2000,1,0.1000,1000,0.0500,200,0.0500,4405.8925,43883.3564,36,7102.7763,45,0.9455
5,0.0000,15,10,0.2000,1,0.3000,1000,0.0500,200,0.0500,4069.6512,48722.8556,43,8933.2696,59,1.1009
7,0.0500,15,10,0.2000,1,0.3000,1000,0.0500,200,0.0500,4069.6512,48722.8556,43,8933.2696,59,1.1721
2,0.0500,15,10,0.2000,1,0.1000,1000,0.0700,200,0.0500,4009.5079,40626.4502,33,7278.7709,42,0.8192
3,0.0000,15,10,0.2000,1,0.1000,1000,0.0700,200,0.0500,4009.5079,40626.4502,33,7278.7709,42,0.9407
8,0.0700,15,10,0.2000,1,0.1000,1000,0.0500,200,0.0500,4009.5079,40626.4502,33,7278.7709,42,0.4536
9,0.0700,15,10,0.2000,1,0.1000,1000,0.0700,200,0.0500,4009.5079,40626.4502,33,7278.7709,42,0.4533
0,0.0500,15,10,0.2000,1,0.3000,1000,0.0700,200,0.0500,3673.2667,45465.9495,40,9109.2641,56,0.8104
6,0.0000,15,10,0.2000,1,0.3000,1000,0.0700,200,0.0500,3673.2667,45465.9495,40,9109.2641,56,1.1552


## Stage 1 results

In [88]:
if best_group is not None and not best_group.empty:
    print(f"Copyable group: {len(best_group)} wallets")
    cols = [c for c in ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
                         "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
            if c in best_group.columns]
    print(best_group[cols].head(15).to_string())

Copyable group: 45 wallets
                                        wallet  opening_roi  opening_copyable_roi  opening_pnl  opening_copyable_pnl  copyable_pnl  buy_roi  num_buckets
0   0xc26e742c5c12cc7609264850de49c77fa41933ca       1.1955                1.0061     674.0874              200.7109        8.9683   1.0553          368
1   0x18c9ca57c5148d23d8b3aef33a1d811677491e8a       0.9771                0.8197     446.7652              151.0402      379.7092   1.1270          221
2   0x76305b2a31e7ec35189650c93e8df2a15b92789d       0.7246                0.7884     490.1535              258.8270      447.4757   1.4334          351
3   0x74957ea27ac4fbdee46d861fdae357859ff67fcf       0.3587                0.7800    1048.9341              302.6835      858.2164   0.5852        13983
4   0xd8798d9aa2c7c05bfefaf6436e8cbd4cd5de7bc3       0.6479                0.4740     709.5858              148.4676      266.7017   0.4606          559
5   0x41f623e0baae41e385875a3759c474cf0622828e       0.

In [89]:
wallet_set = set(best_group["wallet"])
print(f"\nSelected {len(wallet_set)} wallets")
for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


Selected 45 wallets

*** TRAIN copyable group ***
  Open  : wallet_pnl=  56404.56  roi=0.1723  |  copyable_pnl=  13911.97  roi=0.1468
  Total : wallet_pnl= 159273.36  roi=0.1129  |  copyable_pnl=  48428.22  roi=0.0875

*** VAL copyable group ***
  Open  : wallet_pnl=  43883.36  roi=0.1023  |  copyable_pnl=   4405.89  roi=0.0413
  Total : wallet_pnl=  88656.79  roi=0.0669  |  copyable_pnl=   7102.78  roi=0.0167

*** TEST copyable group ***
  Open  : wallet_pnl=  40729.36  roi=0.1131  |  copyable_pnl=   5978.59  roi=0.0843
  Total : wallet_pnl=  63142.09  roi=0.0642  |  copyable_pnl=   4759.86  roi=0.0159


## Save stage 1 result

In [90]:
import json
from datetime import datetime, timezone
from pathlib import Path

wallet_cols = [
    "wallet", "buy_roi", "opening_roi", "opening_pnl",
    "opening_copyable_roi", "opening_copyable_pnl",
    "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "max_drawdown_to_pnl", "top_market_pnl_pct", "market_pnl_hhi",
    "wallet_quality",
]
wallet_records = best_group[[c for c in wallet_cols if c in best_group.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

metadata = {
    "type": "openers",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": len(best_group),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_open_copyable_pnl": float(best_row["open_copyable_pnl"]),
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = Path("stage1_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 result -> {out_path}")

Saved stage 1 result -> stage1_result.json


In [91]:
df = df_test[
    (df_test["wallet"].isin(wallet_set))
    & (df_test["side"] == "BUY") & (df_test["position"] == df_test["quantity"])
    ]
len(df)

19969

In [92]:
df.head()

,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,...,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_notional,roi,copyable_roi
19723,0x04b3f873b003eba3774df1e5e47a370d27ee1b7e,0x080289ccb9c040cbe52aba4d84eef3c6c6595b5bf5a2...,3393266937088080414267704121588513668323492744...,2026-06-29 20:00:26+00:00,BUY,25.6410,25.6410,0.7800,20.0000,25.6410,...,Will the highest temperature in Seoul be 31°C ...,"[Weather, Recurring, Hide From New, Seoul, Dai...",Weather,3393266937088080414267704121588513668323492744...,No,5.6410,20.0000,7.5000,0.2821,0.2821
19726,0x04b3f873b003eba3774df1e5e47a370d27ee1b7e,0x074fbe149aa0511207fb86ff146f2146463af6a39bc9...,6138459099457288793792878400235250262897903784...,2026-07-01 06:21:32+00:00,BUY,28.5714,28.5714,0.7000,20.0000,0.0000,...,Will the highest temperature in Guangzhou be 3...,"[Weather, Recurring, Hide From New, Daily Temp...",Weather,6944993623372826844053576525579327132532144970...,Yes,-20.0000,20.0000,13.0609,-1.0000,-1.0000
19730,0x04b3f873b003eba3774df1e5e47a370d27ee1b7e,0x0de472c55ee41825feb61b982bda0995272b4bb358f9...,2208853734397022070337474786661670775728310001...,2026-07-02 08:01:37+00:00,BUY,19.7300,19.7300,0.6809,13.4334,0.0000,...,Will the highest temperature in Ankara be 34°C...,"[Weather, Recurring, Hide From New, Daily Temp...",Weather,8239625341540113788703346150695628200235336399...,No,-13.4334,13.4334,0.3472,-1.0000,-1.0000
19731,0x04b3f873b003eba3774df1e5e47a370d27ee1b7e,0x0abd8b6d9d018c6416e2baa7d55e1817f6260784e4c1...,2545798153827119295764086684047957501957846825...,2026-07-02 08:05:22+00:00,BUY,27.7486,27.7486,0.7208,20.0000,27.7486,...,Will the highest temperature in Paris be 25°C ...,"[Weather, France, Recurring, Hide From New, Da...",Weather,2545798153827119295764086684047957501957846825...,No,7.7486,20.0000,18.9405,0.3874,0.3874
19732,0x04b3f873b003eba3774df1e5e47a370d27ee1b7e,0x0a7d7857d55d487157edc802c89c0d19a7ed2a55a0fa...,8578583784096522685828829699634699228140047516...,2026-07-02 10:04:37+00:00,BUY,27.0270,27.0270,0.7400,20.0000,27.0270,...,Will the highest temperature in Guangzhou be 3...,"[Weather, Recurring, Hide From New, Daily Temp...",Weather,8578583784096522685828829699634699228140047516...,No,7.0270,20.0000,6.0000,0.3514,0.3514
